[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shriramnag/NeonVido-AI/blob/main/colab/NeonVido_Server.ipynb)

# 🎬 NeonVido-AI (v1.0.006) — Direct Single-File Loader (5.7GB Only, Zero Extra Download)
**निर्देश:** `Runtime > Change runtime type > T4 GPU` चुनें और Play (▶) दबाएं।

In [ ]:
# 1. Colab को सोने से रोकने के लिए Anti-Disconnect JavaScript
from IPython.display import display, Javascript
display(Javascript('''
function KeepColabAlive(){
    const btn = document.querySelector("colab-connect-button") || document.querySelector("#top-toolbar");
    if(btn) { btn.dispatchEvent(new MouseEvent('mousemove', {bubbles: true})); }
}
setInterval(KeepColabAlive, 45000);
'''))

# 2. torchaudio हटाएं और केवल ज़रूरी पैकेज इंस्टॉल करें
!pip uninstall -y -q torchaudio
!pip install -q sentencepiece protobuf diffusers transformers accelerate bitsandbytes fastapi uvicorn nest-asyncio imageio[ffmpeg] huggingface_hub opencv-python-headless peft pillow
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

import os, re, gc, io, time, base64, threading, subprocess, torch, cv2
from PIL import Image
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse
from pydantic import BaseModel
from typing import Optional
from diffusers import LTXPipeline, LTXImageToVideoPipeline
from diffusers.utils import export_to_video
from huggingface_hub import hf_hub_download
import uvicorn

gc.collect()
torch.cuda.empty_cache()

# 3. सीधे आपके Hugging Face से केवल 5.72 GB की सिंगल मॉडल फाइल डाउनलोड करें (कोई 28GB फालतू डाउनलोड नहीं)
print("⏳ 1/2: आपके Hugging Face (Shiv_AI_video_model) से केवल 5.72GB मॉडल फाइल ली जा रही है...")
single_model_path = hf_hub_download(
    repo_id="Shriramnag/Shiv_AI_video_model",
    filename="ltx-video-2b-v0.9.1.safetensors"
)

# 4. लो-मेमोरी में इंजन लोड करें
print("⏳ 2/2: वीडियो इंजन लोड हो रहा है...")
t2v_pipe = LTXPipeline.from_single_file(
    single_model_path,
    torch_dtype=torch.bfloat16
)
t2v_pipe.enable_model_cpu_offload()
t2v_pipe.vae.enable_tiling()

i2v_pipe = LTXImageToVideoPipeline.from_pipe(t2v_pipe)
i2v_pipe.enable_model_cpu_offload()
i2v_pipe.vae.enable_tiling()

print("✅ 100% तैयार! बिना क्रैश हुए आपका मॉडल लोड हो गया है!")

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

@app.get("/ping")
async def ping():
    return {"status": "alive"}

class VideoReq(BaseModel):
    model: str
    prompt: str
    target_minutes: float
    ratio: str
    extend: bool = False
    reference_image: Optional[str] = None
    use_voice: bool = False
    voice_script: str = ""

def extract_last_frame(video_path):
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.set(cv2.CAP_PROP_POS_FRAMES, max(0, total - 1))
    ret, frame = cap.read()
    cap.release()
    if ret:
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        return Image.fromarray(rgb)
    return None

def decode_base64_image(data_url):
    header, encoded = data_url.split(",", 1)
    img_bytes = base64.b64decode(encoded)
    return Image.open(io.BytesIO(img_bytes)).convert("RGB")

@app.post("/generate")
async def generate(req: VideoReq):
    w, h = (480, 704) if req.ratio == "9:16" else (704, 480)
    base_video = "/content/current_master.mp4"

    loops = 1 if req.target_minutes <= 0.2 else min(int((req.target_minutes * 60) // 5), 120)
    if req.extend and os.path.exists(base_video):
        loops = 1

    clip_files = []
    seed_image = None

    if req.extend and os.path.exists(base_video):
        clip_files.append(base_video)
        seed_image = extract_last_frame(base_video)
        print("🔗 पिछले वीडियो के आखिरी फ्रेम से अगला सीन जुड़ रहा है...")
    elif req.reference_image:
        seed_image = decode_base64_image(req.reference_image)
        print("🖼️ रेफरेंस इमेज से वीडियो बन रहा है...")

    prompts_list = [p.strip() for p in req.prompt.split("\n") if p.strip()]

    for i in range(loops):
        current_prompt = prompts_list[i % len(prompts_list)]
        print(f"🎬 क्लिप {i+1}/{loops} बन रही है: {current_prompt}")

        if seed_image is not None:
            seed_resized = seed_image.resize((w, h))
            video_frames = i2v_pipe(
                image=seed_resized,
                prompt=current_prompt,
                negative_prompt="worst quality, inconsistent motion, blurry, jittery, distorted",
                width=w, height=h, num_frames=81, num_inference_steps=20
            ).frames[0]
        else:
            video_frames = t2v_pipe(
                prompt=current_prompt,
                negative_prompt="worst quality, inconsistent motion, blurry, jittery, distorted",
                width=w, height=h, num_frames=81, num_inference_steps=20
            ).frames[0]

        part_path = f"/content/part_{i}.mp4"
        export_to_video(video_frames, part_path, fps=24)
        clip_files.append(part_path)

        seed_image = video_frames[-1]
        gc.collect()
        torch.cuda.empty_cache()

    with open("/content/list.txt", "w") as f:
        for c in clip_files:
            f.write(f"file '{c}'\n")

    subprocess.run(["ffmpeg", "-y", "-f", "concat", "-safe", "0", "-i", "/content/list.txt", "-c", "copy", base_video], check=True)
    print("✅ फाइनल वीडियो तैयार है!")
    return FileResponse(base_video, media_type="video/mp4", filename="NeonVido-AI.mp4")

def run_api():
    uvicorn.run(app, host="127.0.0.1", port=8000)

threading.Thread(target=run_api, daemon=True).start()
time.sleep(2)

tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in tunnel.stdout:
    m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if m:
        print("\n🚀 आपका NEONVIDO-AI LIVE URL:\n" + m.group(0) + "\n")
        break